# Pick out best coeffs

Nudge the best coefficients from the genetic algorithm results to check for better combinations near the current values. Calculate r-squared values to decide which coefficients are the best.

__Inputs:__

+ Best individuals for each generation of each run of the genetic algorithm.
+ MSOA-level admissions and population data for calculating admissions from resulting coefficients.
+ SSNAP-derived coefficients for converting scale factors into probability coefficients.

__Results:__

+ Calculated r-squared values for the fits of calculated to observed admission numbers.

__Method:__

For each run of the genetic algorithm, keep a copy of the best set of coefficients in the final generation.

Calculate the r-squared values associated with each set of coefficients.

Find all combos of coefficients nudged one or two significant figures either way. Then calculate r-squared for those new combos for each depriv quantile separately. Then group the best answer for each quantile to get the final set.

## Code setup

In [1]:
import os
import polars as pl
import numpy as np
from itertools import product

## Load data

### Admissions

In [2]:
path_to_msoa_stats = os.path.join('data', 'msoa_cleaned.csv')

df_stats = pl.read_csv(path_to_msoa_stats)

In [3]:
df_stats.head()

MSOA,admissions,IMD2019Score,All persons,country,good_health,fair health,bad health,prop_good_health,prop_fair health,prop_bad health,MSOA11CD,age_65_proportion,age_70_proportion,age_75_proportion,age_less65_proportion,age_over80_proportion,total_health,age_less65,age_65,age_70,age_75,age_over80,depriv_quantile_min,depriv_quantile_max
str,f64,f64,i64,str,i64,i64,i64,f64,f64,f64,str,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64
"""Adur 001""",14.333333,16.924833,8815,"""E""",6799,1251,474,0.79763,0.146762,0.055608,"""E02006534""",0.0559,0.0528,0.0422,0.7872,0.062,8524,6939.168,492.7585,465.432,371.993,546.53,0.4,0.6
"""Adur 002""",7.333333,6.4704,7263,"""E""",5537,838,259,0.83464,0.126319,0.039041,"""E02006535""",0.0578,0.0774,0.0492,0.7467,0.0692,6634,5423.2821,419.8014,562.1562,357.3396,502.5996,0.8,1.0
"""Adur 003""",9.333333,13.7334,7354,"""E""",5820,969,311,0.819718,0.136479,0.043803,"""E02006536""",0.0609,0.0582,0.0421,0.7729,0.0661,7100,5683.9066,447.8586,428.0028,309.6034,486.0994,0.6,0.8
"""Adur 004""",21.0,26.199857,10582,"""E""",7872,1546,709,0.777328,0.152661,0.070011,"""E02006537""",0.0465,0.0438,0.0367,0.8091,0.0638,10127,8561.8962,492.063,463.4916,388.3594,675.1316,0.2,0.4
"""Adur 005""",13.666667,11.7948,9059,"""E""",7106,1081,339,0.833451,0.126789,0.039761,"""E02006538""",0.0597,0.067,0.0425,0.7643,0.0662,8526,6923.7937,540.8223,606.953,385.0075,599.7058,0.6,0.8


Pick out column names for the health and age proportions:

In [4]:
props_age = [
    'age_less65_proportion', 'age_65_proportion', 'age_70_proportion',
    'age_75_proportion', 'age_over80_proportion'
]
age_numbers = [p.replace('_proportion', '') for p in props_age]

In [5]:
qmin_list = sorted(df_stats['depriv_quantile_min'].unique())

In [6]:
# Names of coeffs:
coeff_names = [f'{a}_q{str(round(q, 1)).replace(".", "")}' for q in qmin_list for a in age_numbers]

coeff_names

['age_less65_q00',
 'age_65_q00',
 'age_70_q00',
 'age_75_q00',
 'age_over80_q00',
 'age_less65_q02',
 'age_65_q02',
 'age_70_q02',
 'age_75_q02',
 'age_over80_q02',
 'age_less65_q04',
 'age_65_q04',
 'age_70_q04',
 'age_75_q04',
 'age_over80_q04',
 'age_less65_q06',
 'age_65_q06',
 'age_70_q06',
 'age_75_q06',
 'age_over80_q06',
 'age_less65_q08',
 'age_65_q08',
 'age_70_q08',
 'age_75_q08',
 'age_over80_q08']

In [7]:
quantile_str_list = sorted(list(set([c.split('_')[-1] for c in coeff_names])))
age_str_list = ['age_less65', 'age_65', 'age_70', 'age_75', 'age_over80']

In [8]:
quantile_str_list

['q00', 'q02', 'q04', 'q06', 'q08']

In [9]:
age_str_list

['age_less65', 'age_65', 'age_70', 'age_75', 'age_over80']

Gather admissions data by deprivation quantile:

In [10]:
admissions_lists = []
x_lists = []

for qmin in qmin_list:
    df_stats_here = df_stats.filter(df_stats['depriv_quantile_min'] == qmin)
    # MSOA data in the same order as those coefficients:
    x_lists_here = [df_stats_here[a] for a in age_numbers]
    admissions_here = df_stats_here['admissions'].to_numpy().tolist()
    # Store:
    admissions_lists.append(admissions_here)
    x_lists.append(x_lists_here)

### Age-admissions coefficients

Starting SSNAP coefficients:

In [11]:
df_pop_admissions = pl.read_csv(os.path.join('outputs', 'ssnap_coeffs.csv'))

In [12]:
df_pop_admissions

Age Groups,population,prop_of_all_pop,count,prop_of_all_admissions,admissions_annual,admissions_annual_boost,prob_stroke_given_age
str,i64,f64,i64,f64,f64,f64,f64
"""Under 65""",45933245,0.816055,38827,0.231449,12942.33333,18737.66819,0.000408
"""65-69""",2796740,0.049687,15324,0.091347,5108.0,7395.26689,0.002644
"""70-74""",2779326,0.049378,21508,0.12821,7169.33333,10379.62674,0.003735
"""75-79""",1940686,0.034478,24150,0.143959,8050.0,11654.63948,0.006005
"""80 and over""",2836964,0.050402,67947,0.405035,22649.0,32790.7987,0.011558


Pick out SSNAP coefficients:

In [13]:
coeffs_ssnap = df_pop_admissions['prob_stroke_given_age'].to_numpy()

coeffs_ssnap

array([0.000408, 0.002644, 0.003735, 0.006005, 0.011558])

Turn them into a dictionary:

In [14]:
labels = ['less65', '65', '70', '75', 'over80']
coeffs_ssnap_dict = dict(zip(labels, df_pop_admissions['prob_stroke_given_age'].to_numpy()))

coeffs_ssnap_dict

{'less65': 0.000408,
 '65': 0.002644,
 '70': 0.003735,
 '75': 0.006005,
 'over80': 0.011558}

Pick out admissions numbers:

In [15]:
dict_admissions_age = dict(zip(df_pop_admissions['Age Groups'], df_pop_admissions['admissions_annual_boost']))

admissions_by_age = list(dict_admissions_age.values())

dict_admissions_age

{'Under 65': 18737.66819,
 '65-69': 7395.26689,
 '70-74': 10379.62674,
 '75-79': 11654.63948,
 '80 and over': 32790.7987}

### Best results from genetic algorithm

Data stored as scale factors for the SSNAP-derived coefficients.

In [16]:
df_best_gens = pl.read_csv(os.path.join('outputs', 'best_inds_deap.csv'))

In [17]:
df_best_gens.head()

dir,# gen,age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,fitness,r2_all,r2_q00,r2_q02,r2_q04,r2_q06,r2_q08
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""randomseed00""",33.0,1.3,1.2,1.332,1.237,1.2,1.026,1.145,1.151,1.041,1.1,1.0,1.0,0.988,0.938,1.0,0.862,0.9,0.945,0.938,0.937,0.8,0.9,0.732,0.832,0.874,238.372,0.584365,0.502719,0.579147,0.616005,0.598245,0.606993
"""randomseed01""",33.0,1.1,1.3,1.305,1.3,1.231,1.1,1.1,1.1,1.023,1.1,1.0,1.0,1.0,1.0,1.0,0.969,0.9,0.9,0.892,0.9,0.837,0.849,0.8,0.8,0.894,238.507,0.575882,0.471584,0.579147,0.616005,0.583824,0.606993
"""randomseed02""",43.0,1.273,1.308,1.175,1.2,1.214,0.981,1.1,1.103,1.1,1.131,0.949,0.987,1.046,1.0,1.0,0.949,0.979,0.896,0.9,0.9,0.844,0.773,0.861,0.8,0.88,238.037,0.58383,0.485431,0.583466,0.616005,0.606683,0.606993
"""randomseed03""",28.0,1.171,1.3,1.327,1.186,1.3,1.1,1.198,1.1,1.1,1.1,1.042,1.0,1.0,1.0,0.971,0.881,0.9,0.894,1.0,0.9,0.8,0.771,0.808,0.7,0.9,239.49,0.585993,0.508469,0.583466,0.631482,0.605116,0.580367
"""randomseed04""",39.0,1.2,1.2,1.25,1.225,1.293,1.1,1.1,1.11,1.069,1.1,0.976,1.019,1.0,1.0,0.961,0.925,0.904,0.9,0.915,0.91,0.8,0.9,0.844,0.8,0.879,238.453,0.591769,0.508469,0.579147,0.631482,0.612783,0.606993


Convert the scale factors to the actual age-deprivation coefficient values using the SSNAP coefficients.

In [18]:
for coeff in coeff_names:
    key = coeff.split('_')[1]
    new_data = df_best_gens[coeff] * coeffs_ssnap_dict[key]
    
    df_best_gens = df_best_gens.with_columns(pl.Series(coeff, new_data))

Round results:

In [19]:
labels = ['less65', '65', '70', '75', 'over80']
round_dict = dict(zip(labels, [4, 3, 3, 3, 3]))  # [5, 4, 4, 4, 4]

for coeff in coeff_names:
    key = coeff.split('_')[1]
    new_data = np.round(df_best_gens[coeff], round_dict[key])
    
    df_best_gens = df_best_gens.with_columns(pl.Series(coeff, new_data))

Drop the fitness measure:

In [20]:
df_best_gens = df_best_gens.drop('fitness')

View results:

In [21]:
df_best_gens.head()

dir,# gen,age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,r2_all,r2_q00,r2_q02,r2_q04,r2_q06,r2_q08
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""randomseed00""",33.0,0.0005,0.003,0.005,0.007,0.014,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.004,0.006,0.011,0.0003,0.002,0.003,0.005,0.01,0.584365,0.502719,0.579147,0.616005,0.598245,0.606993
"""randomseed01""",33.0,0.0004,0.003,0.005,0.008,0.014,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,0.575882,0.471584,0.579147,0.616005,0.583824,0.606993
"""randomseed02""",43.0,0.0005,0.003,0.004,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.003,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,0.58383,0.485431,0.583466,0.616005,0.606683,0.606993
"""randomseed03""",28.0,0.0005,0.003,0.005,0.007,0.015,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.011,0.0004,0.002,0.003,0.006,0.01,0.0003,0.002,0.003,0.004,0.01,0.585993,0.508469,0.583466,0.631482,0.605116,0.580367
"""randomseed04""",39.0,0.0005,0.003,0.005,0.007,0.015,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.011,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,0.591769,0.508469,0.579147,0.631482,0.612783,0.606993


## Recalculate overall fitness

In [22]:
# the goal ('fitness') function to be maximized
def eval_admissions(individual, admissions_lists, x_lists, admissions_by_age):

    # Predictions for each age band across England:
    predictions_by_age = [0.0] * 5
    for i in range(5):
        # Population numbers for areas in this quantile:
        x_lists_here = x_lists[i]
        # Separate prediction for each deprivation quantile:
        coeffs = individual[(i*5):(i*5)+5] #* np.array(coeffs_ssnap)
        yhat_list = predict_admissions_each_age(x_lists_here, coeffs)
        # predictions_lists.append(yhat)
        for j, y in enumerate(yhat_list):
            predictions_by_age[j] += y
    # Divide the admissions by age band by the observed values:
    for j in range(len(predictions_by_age)):
        predictions_by_age[j] /= admissions_by_age[j]
    # Wrongness ratio factor:
    rat = sum(np.abs(np.array(predictions_by_age) - 1.0)) + 1.0
    
    # Predictions for each MSOA:
    predictions_lists = []
    for i in range(5):
        # Population numbers for areas in this quantile:
        x_lists_here = x_lists[i]
        # Separate prediction for each deprivation quantile:
        coeffs = individual[(i*5):(i*5)+5] #* np.array(coeffs_ssnap)
        yhat = predict_admissions(x_lists_here, coeffs)
        predictions_lists.append(yhat)

    # Combine lists:
    observed_all = sum(admissions_lists, [])
    predicted_all = sum(predictions_lists, [])
    # Use only one of the following options for checking the fit:
    sum_sqres = find_square_residuals(predicted_all, observed_all)
    # sum_sqres = find_mean_abs_diff(yhat, admissions)

    # Apply wrongness factor:
    # sum_sqres *= rat
    
    return (sum_sqres, rat, sum_sqres * rat)

In [28]:
list_all_sqr = []
list_all_rat = []
list_all_fit = []

for d in df_best_gens['dir']:
    df = df_best_gens.filter(df_best_gens['dir'] == d)
    # Pick out coefficients:
    coeffs = df[coeff_names].to_numpy().flatten()
    sum_sqres, rat, fitness = eval_admissions(
        coeffs,
        admissions_lists,
        x_lists,
        admissions_by_age
    )
    list_all_sqr.append(sum_sqres)
    list_all_rat.append(rat)
    list_all_fit.append(fitness)

In [29]:
df_best_gens = df_best_gens.with_columns(pl.Series('sum_sqres', list_all_sqr))
df_best_gens = df_best_gens.with_columns(pl.Series('wrong_rat', list_all_rat))
df_best_gens = df_best_gens.with_columns(pl.Series('fitness', list_all_fit))

In [30]:
df_best_gens

dir,# gen,age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,r2_all,r2_q00,r2_q02,r2_q04,r2_q06,r2_q08,sum_sqres,wrong_rat,fitness
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""randomseed00""",33.0,0.0005,0.003,0.005,0.007,0.014,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.004,0.006,0.011,0.0003,0.002,0.003,0.005,0.01,0.584365,0.502719,0.579147,0.616005,0.598245,0.606993,240.103163,1.140725,273.891734
"""randomseed01""",33.0,0.0004,0.003,0.005,0.008,0.014,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,0.575882,0.471584,0.579147,0.616005,0.583824,0.606993,242.540903,1.137745,275.949636
"""randomseed02""",43.0,0.0005,0.003,0.004,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.003,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,0.58383,0.485431,0.583466,0.616005,0.606683,0.606993,240.257537,1.113647,267.56215
"""randomseed03""",28.0,0.0005,0.003,0.005,0.007,0.015,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.011,0.0004,0.002,0.003,0.006,0.01,0.0003,0.002,0.003,0.004,0.01,0.585993,0.508469,0.583466,0.631482,0.605116,0.580367,239.632631,1.085397,260.096504
"""randomseed04""",39.0,0.0005,0.003,0.005,0.007,0.015,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.011,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,0.591769,0.508469,0.579147,0.631482,0.612783,0.606993,237.954979,1.11229,264.674883
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""randomseed95""",35.0,0.0005,0.003,0.005,0.007,0.014,0.0005,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.006,0.01,0.0003,0.002,0.003,0.005,0.01,0.583789,0.502719,0.569427,0.616005,0.605116,0.606993,240.269379,1.101715,264.708478
"""randomseed96""",28.0,0.0005,0.003,0.005,0.009,0.014,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.011,0.0004,0.002,0.003,0.006,0.01,0.0004,0.002,0.003,0.005,0.01,0.590145,0.506488,0.579147,0.631482,0.605116,0.608414,238.427782,1.162697,277.219247
"""randomseed97""",46.0,0.0005,0.003,0.005,0.007,0.015,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.003,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,0.587962,0.508469,0.583466,0.616005,0.606683,0.606993,239.061938,1.075607,257.13671


## Recalculate "fitness" for each depriv separately

In [25]:
def predict_admissions(x_lists, coeffs):
    """
    x_lists: np.array.
    coeffs: np.array.
    
    Have to have same number of coeffs as x_lists.
    """
    # Predicted admissions:
    # yhat = (x_lists * coeffs.reshape(len(coeffs), 1)).sum(axis=0)
    yhat = sum(
        [x_lists[i] * coeffs[i] for i in range(len(coeffs))]
    )
    return yhat.to_numpy().tolist()

In [24]:
def predict_admissions_each_age(x_lists, coeffs):
    """
    x_lists: np.array.
    coeffs: np.array.
    
    Have to have same number of coeffs as x_lists.
    """
    # Predicted admissions:
    yhat = (
        [(x_lists[i] * coeffs[i]).sum() for i in range(len(coeffs))]
    )
    return yhat

Goodness check option 1: This calculates the sum of the square of the differences between predicted and actual admission numbers:

In [26]:
def find_square_residuals(yhat, y):
    # Difference from actual:
    sqres = (np.array(yhat) - np.array(y))**2.0
    # Sum of differences:
    sum_sqres = sqres.sum()
    return np.sqrt(sum_sqres)

In [27]:
# the goal ('fitness') function to be maximized
def eval_admissions_one_depriv(
        coeffs,
        admissions_here,
        x_lists_here,
        admissions_by_age
    ):
    """
        coeffs,
        admissions_here,
        x_lists_here, - Population numbers for areas in this quantile.
        admissions_by_age
    """
    # Predictions for each age band across England:
    predictions_by_age = [0.0] * 5
    # Predict admissions:
    yhat_list = predict_admissions_each_age(x_lists_here, coeffs)
    for j, y in enumerate(yhat_list):
        predictions_by_age[j] += y
    # Divide the admissions by age band by the observed values:
    for j in range(len(predictions_by_age)):
        predictions_by_age[j] /= admissions_by_age[j]   # NEED TO CHANGE THIS VALUE - shouldn't be comparing admissions for this age+depriv with total england admissions for this age
    # Wrongness ratio factor:
    rat = sum(np.abs(np.array(predictions_by_age) - 1.0)) + 1.0
    
    # Predictions for each MSOA:
    yhat = predict_admissions(x_lists_here, coeffs)
    # Use only one of the following options for checking the fit:
    sum_sqres = find_square_residuals(yhat, admissions_here)
    # sum_sqres = find_mean_abs_diff(yhat, admissions)
    
    return (sum_sqres, rat, sum_sqres * rat)

In [31]:
list_sqr = [[], [], [], [], []]  # getting errors with [[]]*5
list_rat = [[], [], [], [], []]
list_fit = [[], [], [], [], []]

for d in df_best_gens['dir']:
    df = df_best_gens.filter(df_best_gens['dir'] == d)

    for i, quantile_str in enumerate(quantile_str_list):
        # Pick out coefficients:
        coeff_cols = [f'{a}_{quantile_str}' for a in age_str_list]
        coeffs = df[coeff_cols].to_numpy().flatten()
        sum_sqres, rat, fitness = eval_admissions_one_depriv(
            coeffs,
            admissions_lists[i],
            x_lists[i],
            admissions_by_age
        )
        list_sqr[i].append(sum_sqres)
        list_rat[i].append(rat)
        list_fit[i].append(fitness)

In [32]:
for i, quantile_str in enumerate(quantile_str_list):
    df_best_gens = df_best_gens.with_columns(pl.Series(f'sum_sqres_{quantile_str}', list_sqr[i]))
    df_best_gens = df_best_gens.with_columns(pl.Series(f'wrong_rat_{quantile_str}', list_rat[i]))
    df_best_gens = df_best_gens.with_columns(pl.Series(f'fitness_{quantile_str}', list_fit[i]))

In [33]:
df_best_gens

dir,# gen,age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,r2_all,r2_q00,r2_q02,r2_q04,r2_q06,r2_q08,sum_sqres,wrong_rat,fitness,sum_sqres_q00,wrong_rat_q00,fitness_q00,sum_sqres_q02,wrong_rat_q02,fitness_q02,sum_sqres_q04,wrong_rat_q04,fitness_q04,sum_sqres_q06,wrong_rat_q06,fitness_q06,sum_sqres_q08,wrong_rat_q08,fitness_q08
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""randomseed00""",33.0,0.0005,0.003,0.005,0.007,0.014,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.004,0.006,0.011,0.0003,0.002,0.003,0.005,0.01,0.584365,0.502719,0.579147,0.616005,0.598245,0.606993,240.103163,1.140725,273.891734,111.220039,5.011406,557.36881,108.40745,5.026433,544.902762,108.96863,4.906511,534.655808,105.838415,4.939217,522.758847,102.232696,5.102575,521.649976
"""randomseed01""",33.0,0.0004,0.003,0.005,0.008,0.014,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,0.575882,0.471584,0.579147,0.616005,0.583824,0.606993,242.540903,1.137745,275.949636,114.648926,5.040112,577.843427,108.40745,5.026433,544.902762,108.96863,4.906511,534.655808,107.721177,5.062114,545.296864,102.232696,5.102575,521.649976
"""randomseed02""",43.0,0.0005,0.003,0.004,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.003,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,0.58383,0.485431,0.583466,0.616005,0.606683,0.606993,240.257537,1.113647,267.56215,113.136822,5.050766,571.427561,107.849722,4.996719,538.894793,108.96863,4.906511,534.655808,104.721032,4.978835,521.388713,102.232696,5.102575,521.649976
"""randomseed03""",28.0,0.0005,0.003,0.005,0.007,0.015,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.011,0.0004,0.002,0.003,0.006,0.01,0.0003,0.002,0.003,0.004,0.01,0.585993,0.508469,0.583466,0.631482,0.605116,0.580367,239.632631,1.085397,260.096504,110.575162,4.999076,552.773594,107.849722,4.996719,538.894793,106.750111,4.924622,525.703917,104.929413,5.021888,526.943721,105.639012,5.143093,543.311215
"""randomseed04""",39.0,0.0005,0.003,0.005,0.007,0.015,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.011,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,0.591769,0.508469,0.579147,0.631482,0.612783,0.606993,237.954979,1.11229,264.674883,110.575162,4.999076,552.773594,108.40745,5.026433,544.902762,106.750111,4.924622,525.703917,103.905822,5.042072,523.900606,102.232696,5.102575,521.649976
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""randomseed95""",35.0,0.0005,0.003,0.005,0.007,0.014,0.0005,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.006,0.01,0.0003,0.002,0.003,0.005,0.01,0.583789,0.502719,0.569427,0.616005,0.605116,0.606993,240.269379,1.101715,264.708478,111.220039,5.011406,557.36881,109.65222,4.975186,545.540197,108.96863,4.906511,534.655808,104.929413,5.021888,526.943721,102.232696,5.102575,521.649976
"""randomseed96""",28.0,0.0005,0.003,0.005,0.009,0.014,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.011,0.0004,0.002,0.003,0.006,0.01,0.0004,0.002,0.003,0.005,0.01,0.590145,0.506488,0.579147,0.631482,0.605116,0.608414,238.427782,1.162697,277.219247,110.797733,4.962887,549.876671,108.40745,5.026433,544.902762,106.750111,4.924622,525.703917,104.929413,5.021888,526.943721,102.047668,5.056976,516.052606
"""randomseed97""",46.0,0.0005,0.003,0.005,0.007,0.015,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.003,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,0.587962,0.5084

## Test: can other combos be better?

From the table above we can see that the best overall combo is not made up of the best combo for each deprivation quantile:

In [34]:
r2_cols = [c for c in df_best_gens.columns if c.startswith('r2')]

d1 = df_best_gens.sort('r2_all', descending=True)[0][r2_cols]
d2 = df_best_gens[r2_cols].max()

d1 = d1.with_columns(pl.Series('coeff_combo', ['max_r2_overall']))
d2 = d2.with_columns(pl.Series('coeff_combo', [f'max_r2_each_depriv']))

display(pl.concat((d1, d2)))

r2_all,r2_q00,r2_q02,r2_q04,r2_q06,r2_q08,coeff_combo
f64,f64,f64,f64,f64,f64,str
0.593483,0.509119,0.583466,0.634746,0.612783,0.606993,"""max_r2_overall"""
0.593483,0.509119,0.583846,0.634746,0.616007,0.62481,"""max_r2_each_depriv"""


The second row of the table has the best r-squared for each deprivation quantile. Its values are higher for the second and the two most-deprived quantiles than in the overall best combo.

View the best combo for these deprivation quantiles:

In [35]:
for q in ['q02', 'q06', 'q08']:
    coeffs_q = [c for c in df_best_gens.columns if q in c]
    mask = df_best_gens.sort('r2_all', descending=True)[f'r2_{q}'] == df_best_gens[f'r2_{q}'].max()

    d1 = df_best_gens.sort('r2_all', descending=True)[0][coeffs_q]
    d2 = df_best_gens.sort('r2_all', descending=True).filter(mask)[0][coeffs_q]
    d1 = d1.with_columns(pl.Series('coeff_combo', ['best_overall']))
    d2 = d2.with_columns(pl.Series('coeff_combo', [f'best_{q}']))

    display(pl.concat((d1, d2)))

age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,r2_q02,sum_sqres_q02,wrong_rat_q02,fitness_q02,coeff_combo
f64,f64,f64,f64,f64,f64,f64,f64,f64,str
0.0004,0.003,0.004,0.007,0.013,0.583466,107.849722,4.996719,538.894793,"""best_overall"""
0.0004,0.003,0.004,0.006,0.014,0.583846,107.800573,5.011427,540.234733,"""best_q02"""


age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,r2_q06,sum_sqres_q06,wrong_rat_q06,fitness_q06,coeff_combo
f64,f64,f64,f64,f64,f64,f64,f64,f64,str
0.0004,0.002,0.003,0.005,0.011,0.612783,103.905822,5.042072,523.900606,"""best_overall"""
0.0004,0.002,0.003,0.006,0.011,0.616007,103.472366,5.001845,517.552785,"""best_q06"""


age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,r2_q08,sum_sqres_q08,wrong_rat_q08,fitness_q08,coeff_combo
f64,f64,f64,f64,f64,f64,f64,f64,f64,str
0.0003,0.002,0.003,0.005,0.01,0.606993,102.232696,5.102575,521.649976,"""best_overall"""
0.0003,0.002,0.003,0.005,0.011,0.62481,99.88842,5.08167,507.600023,"""best_q08"""


Manually adjust the values in the best combo to recreate the best r-squared for each deprivation quantile.

In [36]:
# Pick out coeffs:
coeffs = df_best_gens.filter(df_best_gens['r2_all'] == df_best_gens['r2_all'].max())[coeff_names].to_numpy().flatten()
# Update some:
coeffs[coeff_names.index('age_75_q02')] = 0.006
coeffs[coeff_names.index('age_over80_q02')] = 0.014
coeffs[coeff_names.index('age_75_q06')] = 0.006
coeffs[coeff_names.index('age_over80_q08')] = 0.011

Check that coeffs still increase with age (across) and deprivation level (upwards):

In [37]:
coeffs.reshape(5, 5)

array([[0.0005, 0.004 , 0.005 , 0.007 , 0.014 ],
       [0.0004, 0.003 , 0.004 , 0.006 , 0.014 ],
       [0.0004, 0.002 , 0.004 , 0.006 , 0.012 ],
       [0.0004, 0.002 , 0.003 , 0.006 , 0.011 ],
       [0.0003, 0.002 , 0.003 , 0.005 , 0.011 ]])

What's the effect on overall R^2?

In [41]:
list_r2 = calculate_many_rsquared(coeffs, x_lists, coeffs_ssnap, admissions_lists)
# List contains R^2 for each quantile in turn, then R^2 of all.

# Best R^2 from all genetic algorithm results:
r2_best = df_best_gens['r2_all'].max()

print(f'New r-squared: {list_r2[-1]:.4f}')
print(f'Old r-squared: {r2_best:.4f}')

New r-squared: 0.5976
Old r-squared: 0.5935


In [42]:
sum_sqres, rat, fitness = eval_admissions(
    coeffs,
    admissions_lists,
    x_lists,
    admissions_by_age
)

# Best R^2 from all genetic algorithm results:
fitness_best = df_best_gens['fitness'].min()

print(f'New fitness: {fitness:.4f}')
print(f'Old fitness: {fitness_best:.4f}')

New fitness: 271.9798
Old fitness: 252.5461


The overall r-squared has increased with this manual changing of the coefficients.

It is worth trying more combinations of coefficients near the found values to check whether any small adjustments can make better results. The genetic algorithm isn't likely to have tried every good combination by chance, and there could be other better options yet to be discovered.

## Grid near best combo

Take the best coefficients. For each deprivation quantile, nudge the best coefficient one or two clicks either way and calculate the new fitness scores.

In [43]:
def calculate_new_combos_fitnesses(quantile_str, start_coeffs, best_coeff_cols):
    """
    Find all combinations of nudged options for one starting set of 5
    coeffs. 5 options x 5 coeffs = 3125 nudged options.
    """
    qi = quantile_str_list.index(quantile_str)
    
    # Add on the offsets to make the new options:
    # Set up each value to be nudged one or two sig figs up or down.
    new_options = [
        [round(start_coeffs[0] + i*1e-4, 4) for i in range(-2, 3)],  # age less 65
        [round(start_coeffs[1] + i*1e-3, 3) for i in range(-2, 3)],  # age 65-69
        [round(start_coeffs[2] + i*1e-3, 3) for i in range(-2, 3)],  # age 70-74
        [round(start_coeffs[3] + i*1e-3, 3) for i in range(-2, 3)],  # age 75-79
        [round(start_coeffs[4] + i*1e-3, 3) for i in range(-2, 3)],  # age over 80
    ]
    # Generate all combinations of these new parameters.
    # Each element of this list is a tuple of five new coeffs.
    all_new_option_combos = list(product(*new_options))
    # Only keep combinations where coefficients increase with age band:
    mask = (np.diff(all_new_option_combos, axis=1) >= 0).all(axis=1)
    all_new_option_combos = np.array(all_new_option_combos)[mask]

    # Store results in here:
    list_r2 = []
    # Calculate r-squared:
    for c, coeffs in enumerate(all_new_option_combos):
        # Population numbers for areas in this quantile:
        x_lists_here = x_lists[qi]
        # Separate prediction for each deprivation quantile:
        yhat = predict_admissions(x_lists_here, np.array(list(coeffs)))
        # Compare with the observed admissions to calculate r-squared:
        observed_here = admissions_lists[qi]
        r2 = calculate_rsquared(observed_here, yhat)
        # Store result:
        list_r2.append(r2)
    
    # Place new coeff combos into dataframe:
    df_new_combos = pl.DataFrame(all_new_option_combos, schema=best_coeff_cols, orient='row')
    # Make a new column with the r-squared values:
    df_new_combos = df_new_combos.with_columns(pl.Series(f'r2_{quantile_str}', list_r2))
    return df_new_combos

Pick out the best set of coefficients:

In [44]:
df_best = df_best_gens.filter(df_best_gens['r2_all'] == df_best_gens['r2_all'].max())

Find every nudged variation on the best coefficients and calculate the new r-squared for that deprivation quantile.

In [45]:
new_combo_df_dict = {}

for quantile_str in quantile_str_list:
    
    best_coeff_cols = [f'{a}_{quantile_str}' for a in age_str_list]
    start_coeffs = df_best[best_coeff_cols].to_numpy().flatten()
    
    df_new_combos = calculate_new_combos_fitnesses(quantile_str, start_coeffs, best_coeff_cols)
    new_combo_df_dict[quantile_str] = df_new_combos

View the results:

In [46]:
for quantile_str in list(new_combo_df_dict.keys()):
    print(quantile_str)
    # Print the best coefficients for comparison:
    cols = [f'{a}_{quantile_str}' for a in age_str_list] + [f'r2_{quantile_str}', 'r2_all']
    print('Coeffs in best overall combo:')
    display(df_best[cols])
    # Print the nudged coeffs from best to worst:
    print('Nudged coeffs:')
    display(new_combo_df_dict[quantile_str].sort(f'r2_{quantile_str}', descending=True))
    print('\n'*2)

q00
Coeffs in best overall combo:


age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,r2_q00,r2_all
f64,f64,f64,f64,f64,f64,f64
0.0005,0.004,0.005,0.007,0.014,0.509119,0.593483


Nudged coeffs:


age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,r2_q00
f64,f64,f64,f64,f64,f64
0.0004,0.006,0.006,0.006,0.014,0.510425
0.0005,0.005,0.005,0.005,0.014,0.510094
0.0004,0.006,0.006,0.007,0.013,0.510005
0.0004,0.005,0.006,0.006,0.015,0.509989
0.0004,0.005,0.005,0.006,0.016,0.509762
…,…,…,…,…,…
0.0007,0.006,0.007,0.009,0.015,-0.186458
0.0003,0.002,0.003,0.006,0.012,-0.191596
0.0007,0.006,0.007,0.008,0.016,-0.222319





q02
Coeffs in best overall combo:


...

age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,r2_q06
f64,f64,f64,f64,f64,f64
0.0003,0.002,0.002,0.006,0.013,0.624681
0.0003,0.001,0.003,0.006,0.013,0.624445
0.0003,0.001,0.002,0.007,0.013,0.624237
0.0003,0.002,0.003,0.005,0.013,0.623275
0.0003,0.003,0.003,0.003,0.013,0.622883
…,…,…,…,…,…
0.0006,0.003,0.005,0.007,0.013,-0.248632
0.0006,0.004,0.005,0.006,0.013,-0.294663
0.0002,0.001,0.001,0.004,0.009,-0.296746





q08
Coeffs in best overall combo:


age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,r2_q08,r2_all
f64,f64,f64,f64,f64,f64,f64
0.0003,0.002,0.003,0.005,0.01,0.606993,0.593483


Nudged coeffs:


age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,r2_q08
f64,f64,f64,f64,f64,f64
0.0001,0.003,0.004,0.004,0.012,0.634189
0.0002,0.001,0.004,0.005,0.012,0.633741
0.0002,0.002,0.003,0.005,0.012,0.633663
0.0001,0.002,0.003,0.007,0.012,0.633617
0.0001,0.003,0.003,0.006,0.012,0.633616
…,…,…,…,…,…
0.0001,0.001,0.001,0.003,0.009,-0.383149
0.0005,0.004,0.005,0.007,0.012,-0.385835
0.0001,0.001,0.002,0.003,0.008,-0.403286


For all deprivation quantiles, the sets of nudged coeffs contain an option with higher r-squared than the one in the best overall combo.

Gather the coefficients that are best for each deprviation quantile:

In [47]:
best_combos = []
best_combo_cols = []

for q, quantile_str in enumerate(list(new_combo_df_dict.keys())):
    # display(df_best_gens.filter(df_best_gens['r2_all'] == df_best_gens['r2_all'].max())[[f'{a}_{quantile_str}' for a in age_str_list] + [f'r2_{quantile_str}', 'r2_all']])
    df = new_combo_df_dict[quantile_str].sort(f'r2_{quantile_str}', descending=True)
    df = df[0] if q < 4 else df[3]
    cols = [c for c in df.columns if c.startswith('age')]
    best_combo_cols += cols
    best_combos += list(df[cols].to_numpy().flatten())

Recalculate total r-squared:

In [48]:
list_r2 = calculate_many_rsquared(best_combos, x_lists, coeffs_ssnap, admissions_lists)
# List contains R^2 for each quantile in turn, then R^2 of all.

# Best R^2 from all genetic algorithm results:
r2_best = df_best_gens['r2_all'].max()

print(f'New r-squared: {list_r2[-1]:.4f}')
print(f'Old r-squared: {r2_best:.4f}')

New r-squared: 0.6040
Old r-squared: 0.5935


In [49]:
sum_sqres, rat, fitness = eval_admissions(
    best_combos,
    admissions_lists,
    x_lists,
    admissions_by_age
)

# Best R^2 from all genetic algorithm results:
fitness_best = df_best_gens['fitness'].min()

print(f'New fitness: {fitness:.4f}')
print(f'Old fitness: {fitness_best:.4f}')

New fitness: 402.5061
Old fitness: 252.5461


R-squared has improved using these nudged coefficients.

Check that the complete set of coefficients meet the requirements of increasing with age and deprivation:

In [50]:
np.array(best_combos).reshape(5, 5)

array([[0.0004, 0.006 , 0.006 , 0.006 , 0.014 ],
       [0.0004, 0.003 , 0.004 , 0.006 , 0.014 ],
       [0.0003, 0.002 , 0.002 , 0.007 , 0.014 ],
       [0.0003, 0.002 , 0.002 , 0.006 , 0.013 ],
       [0.0001, 0.002 , 0.003 , 0.007 , 0.012 ]])

These requirements are not met! The values break the pattern in a handful of places.

Instead apply the same method to all of the 100 sets of coefficients from the genetic algorithm results. See if any combination of those resulting nudged values both improves the overall r-squared and meets the increasing criteria.

## Nudge all 100 fits

Repeat the above process for all of the 100 best sets of coefficents.

Store the results in a dictionary:

In [39]:
new_combo_100_df_dict = {}

for quantile_str in quantile_str_list:
    # Pick out best r-squared for this depriv quantile only
    # in the starting 100 sets of fits:
    r2_col = f'r2_{quantile_str}'
    r2_best_q = df_best_gens[r2_col].max() 
    # Set up column names for this depriv quantile's coeffs:
    best_coeff_cols = [f'{a}_{quantile_str}' for a in age_str_list]
    # Run through each of the 100 sets of starting coefficients:
    for i in range(len(df_best_gens)):
        # Progress tracker:
        print(f'{quantile_str}: {i+1:3d} out of {len(df_best_gens)}', end='\r')
        # Pick out the starting coefficients:
        start_coeffs = df_best_gens[i][best_coeff_cols].to_numpy().flatten()
        # Find all nudged combinations and their r-squared values:
        df = calculate_new_combos_fitnesses(quantile_str, start_coeffs, best_coeff_cols)
        # Only keep combos with decent r-squared, at least 95% of
        # the value from the best of the 100 starting fits.
        df = df.filter(df[r2_col] >= (0.95 * r2_best_q))
        if i == 0:
            # Keep a copy of this data: 
            df_new_combos = df
        else:
            # Join these onto the existing results.
            df_new_combos = pl.concat((df_new_combos, df), how='vertical')
        # Remove duplicate columns:
        df_new_combos = df_new_combos.unique()
    # Once all 100 starting combos have run, store the results for
    # this deprivation quantile:
    new_combo_100_df_dict[quantile_str] = df_new_combos

q00
q02 out of 100
q04 out of 100
q06 out of 100
q08 out of 100


View the results. Each deprivation quantile has its own entry in the dictionary:

In [40]:
for quantile_str in list(new_combo_df_dict.keys()):
    display(new_combo_100_df_dict[quantile_str].sort(f'r2_{quantile_str}', descending=True))

age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,r2_q00
f64,f64,f64,f64,f64,f64
0.0004,0.006,0.006,0.006,0.014,0.510425
0.0005,0.005,0.005,0.005,0.014,0.510094
0.0004,0.006,0.006,0.007,0.013,0.510005
0.0004,0.005,0.006,0.006,0.015,0.509989
0.0004,0.005,0.005,0.006,0.016,0.509762
…,…,…,…,…,…
0.0004,0.005,0.005,0.01,0.015,0.483708
0.0004,0.004,0.007,0.01,0.014,0.483701
0.0003,0.003,0.007,0.007,0.016,0.483695


age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,r2_q02
f64,f64,f64,f64,f64,f64
0.0004,0.004,0.004,0.004,0.014,0.584399
0.0004,0.003,0.004,0.004,0.015,0.584127
0.0004,0.003,0.004,0.006,0.014,0.583846
0.0004,0.004,0.004,0.006,0.013,0.583672
0.0004,0.003,0.003,0.007,0.014,0.583667
…,…,…,…,…,…
0.0004,0.001,0.005,0.007,0.016,0.554771
0.0002,0.003,0.004,0.006,0.016,0.554734
0.0006,0.001,0.005,0.007,0.011,0.554707


age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,r2_q04
f64,f64,f64,f64,f64,f64
0.0003,0.002,0.002,0.007,0.014,0.646679
0.0003,0.002,0.003,0.006,0.014,0.646457
0.0003,0.001,0.003,0.007,0.014,0.646105
0.0003,0.003,0.003,0.004,0.014,0.645803
0.0003,0.002,0.004,0.004,0.014,0.645713
…,…,…,…,…,…
0.0004,0.003,0.004,0.006,0.009,0.603425
0.0005,0.004,0.004,0.006,0.009,0.603402
0.0005,0.002,0.003,0.006,0.01,0.603252


age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,r2_q06
f64,f64,f64,f64,f64,f64
0.0003,0.002,0.002,0.006,0.013,0.624681
0.0003,0.001,0.003,0.006,0.013,0.624445
0.0003,0.001,0.002,0.007,0.013,0.624237
0.0003,0.002,0.003,0.005,0.013,0.623275
0.0003,0.003,0.003,0.003,0.013,0.622883
…,…,…,…,…,…
0.0005,0.003,0.005,0.005,0.008,0.585325
0.0003,0.001,0.004,0.008,0.009,0.58531
0.0003,0.002,0.005,0.007,0.008,0.58531


age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,r2_q08
f64,f64,f64,f64,f64,f64
0.0001,0.003,0.003,0.004,0.013,0.636718
0.0001,0.002,0.004,0.004,0.013,0.636306
0.0002,0.002,0.002,0.005,0.013,0.635773
0.0002,0.001,0.003,0.005,0.013,0.635698
0.0002,0.002,0.003,0.004,0.013,0.635487
…,…,…,…,…,…
0.0002,0.003,0.004,0.005,0.012,0.594071
0.0004,0.001,0.005,0.006,0.009,0.593988
0.0003,0.001,0.002,0.007,0.013,0.593886


All of the deprivation quantiles now have fits with higher r-squared than the best set from the genetic algorithm output.

Gather the best combos for all depriv quantile:

In [42]:
# Store results in here as a single list of 25 values:
best_100_combos = []
# Sanity check that columns are in expected order:
best_100_combo_cols = []

for q, quantile_str in enumerate(list(new_combo_100_df_dict.keys())):
    # Sort nudged values from best to worst r-squared:
    df = new_combo_100_df_dict[quantile_str].sort(f'r2_{quantile_str}', descending=True)
    # Pick out top row (best result)
    df = df[0]
    # Keep a copy of the expected column names:
    cols = [c for c in df.columns if c.startswith('age')]
    best_100_combo_cols += cols
    # Join these five coefficients to the end of the others:
    best_100_combos += list(df[cols].to_numpy().flatten())

Check that it meets the requirements of increasing with age and deprivation:

In [44]:
np.array(best_100_combos).reshape(5, 5)

array([[0.0004, 0.006 , 0.006 , 0.006 , 0.014 ],
       [0.0004, 0.004 , 0.004 , 0.004 , 0.014 ],
       [0.0003, 0.002 , 0.002 , 0.007 , 0.014 ],
       [0.0003, 0.002 , 0.002 , 0.006 , 0.013 ],
       [0.0001, 0.003 , 0.003 , 0.004 , 0.013 ]])

It doesn't work!

Instead, try a few combinations of "best" results. For each deprivation quantile in turn, use its best results and then select results for the other quantiles that work around it.

The following functions search an array of coefficients to find sets that work with a given set of coefficients. They find either coefficients that always increase or stay the same as the reference set, or that always decrease or stay the same.

In [45]:
def pick_conditions_met(combo, all_combos, pick_more=True):
    """
    Find options where all coeffs are more than/same as fixed values.

    Inputs
    ------
    combo      - list. Fixed coefficients.
    all_combos - np.array. One row per set of coefficients.
                 Assume sorted from best to worst r-squared.
    Returns
    -------
    list. The selected coefficients from the big list.
    """
    # List of masks, one mask per age band.
    # Masks are for whether the coeffs for this age band
    # are more or equal to the fixed value for this age band.
    if pick_more:
        masks = [(all_combos[:, p] >= combo[p]) for p in range(len(combo))]
    else:
        masks = [(all_combos[:, p] <= combo[p]) for p in range(len(combo))]
    # Gather masks:
    masks = np.vstack(masks).T
    # Check where condition is met for all age bands:
    valid_mask = masks.all(axis=1)
    # Select this index "pick", the first in the list where the
    # condition is met for all age bands.
    pick = np.where(valid_mask == True)[0][0]
    # Return a list of the coefficients in this row of the table:
    return all_combos[pick]

Pick out combos:

In [46]:
def pick_combos(fixed_combo, new_combo_100_df_dict, q):
    """
    Pick combos of all good coeffs that meet rules.

    Fix the coefficients for the qth quantile in the list.
    Then pick coefficients for the adjacent quantiles that meet rules
    and continue until all quantiles have been picked.
    Picked coefficients must increase with age band and with level
    of deprivation.

    Inputs
    ------
    fixed_combo           - list. Five coefficients that are fixed.
    new_combo_100_df_dict - dict. One entry per deprivation quantile.
                            Each entry is a dataframe of coefficients
                            and their r-squared value.
    q                     - int. Index of depriv quantile for the
                            fixed combo.

    Returns
    -------
    best_combo - list. Length 25, picked coefficients.
    """
    def gather_coeff_combos(p):
        """
        Change big dataframe of coeff combos to array for picking.
        """
        # Find the name of the r-squared column for the next combo:
        p_str = list(new_combo_100_df_dict.keys())[p]
        # Pick out the coefficients and r-squared for this combo:
        arr = new_combo_100_df_dict[p_str].sort(f'r2_{p_str}', descending=True).to_numpy()
        # Cut off r-squared column:
        arr = arr[:, :-1]
        return arr
        
    # Set up coeff storage.
    # Store the coeffs for each quantile in their own list in order
    # in the following list of lists. Start with empty lists...
    best_combo = [[]] * 5
    # ... fill in one list of coeffs...
    best_combo[q] = list(fixed_combo)
    # ... then use that filled list to select the adjacent lists.
    for p in range(q+1, 5):
        # Find the combos where values should be less than or same as the
        # starting values.
        arr = gather_coeff_combos(p)
        # Find the best coeff combo that meets the decreasing criteria:
        picked_coeffs = pick_conditions_met(best_combo[p-1], arr, pick_more=False)
        # Store result:
        best_combo[p] = list(picked_coeffs)
    for p in range(q-1, -1, -1):
        # Find the combos where values should be more than or same as the
        # starting values.
        arr = gather_coeff_combos(p)
        # Find the best coeff combo that meets the increasing criteria:
        picked_coeffs = pick_conditions_met(best_combo[p+1], arr)
        # Store result:
        best_combo[p] = list(picked_coeffs)
    # Return a single flat list of coeffs, length 25.
    best_combo = sum(best_combo, [])
    return best_combo

Run this function for each quantile in turn. For each loop, a different quantile's coefficients are fixed first. Then the other quantiles' coefficients have to work around those fixed values.

In [ ]:
best_picked_combos = {}

for q, quantile_str in enumerate(list(new_combo_100_df_dict.keys())):
    # Pick out a list of five coefficients for this depriv quantile:
    fixed_combo = (new_combo_100_df_dict[quantile_str]
                   .sort(f'r2_{quantile_str}', descending=True)[0]
                   .to_numpy().flatten()[:-1])
    # Pick out coeffs for the other quantiles that work well with this:
    best_picked_combos[quantile_str] = pick_combos(
        fixed_combo, new_combo_100_df_dict, q)

Calculate their r-squared values:

In [108]:
best_picked_combos_r2s = {}

for quantile_str, best_combo in best_picked_combos.items():
    # Calculate R^2:
    list_r2 = calculate_many_rsquared(best_combo, x_lists, coeffs_ssnap, admissions_lists)
    # Output list is R^2 for each quantile in turn, then R^2 of all.
    # Round values:
    list_r2 = [round(r, 5) for r in list_r2]
    # Store the overall R^2 in here:
    best_picked_combos_r2s[quantile_str] = list_r2

Rearrange the picked coefficients into dataframes:

In [108]:
best_picked_combos_dfs = {}
age_label_list = ['less65', '65-70', '70-75', '75-80', 'over80']

for quantile_str, best_combo in best_picked_combos.items():
    # Make a dataframe:
    data = np.array(best_combo).reshape(5, 5)
    # Add deprivation column:
    data = np.hstack((np.array(quantile_str_list).reshape(5, 1), data))
    df = pl.DataFrame(data, schema=['depriv_quantile_min'] + age_label_list)
    for col in age_label_list:
        df = df.with_columns(pl.col(col).cast(float))
    # Place r-squared results in the dataframe:
    list_r2 = best_picked_combos_r2s[quantile_str]
    df = df.with_columns(pl.Series('r2', list_r2[:-1]))
    
    best_picked_combos_dfs[quantile_str] = df

View the results:

In [109]:
for quantile_str, df in best_picked_combos_dfs.items():
    print(f'First fixed: {quantile_str}')
    print(f'Overall R^2: {best_picked_combos_r2s[quantile_str][-1]:.5f}')
    display(df)
    print('')

First fixed: q00
Overall R^2: 0.60357


depriv_quantile_min,less65,65-70,70-75,75-80,over80,r2
str,f64,f64,f64,f64,f64,f64
"""q00""",0.0004,0.006,0.006,0.006,0.014,0.51042
"""q02""",0.0004,0.004,0.004,0.004,0.014,0.5844
"""q04""",0.0003,0.003,0.003,0.004,0.014,0.6458
"""q06""",0.0003,0.003,0.003,0.003,0.013,0.62288
"""q08""",0.0002,0.003,0.003,0.003,0.013,0.63398



First fixed: q02
Overall R^2: 0.60357


depriv_quantile_min,less65,65-70,70-75,75-80,over80,r2
str,f64,f64,f64,f64,f64,f64
"""q00""",0.0004,0.006,0.006,0.006,0.014,0.51042
"""q02""",0.0004,0.004,0.004,0.004,0.014,0.5844
"""q04""",0.0003,0.003,0.003,0.004,0.014,0.6458
"""q06""",0.0003,0.003,0.003,0.003,0.013,0.62288
"""q08""",0.0002,0.003,0.003,0.003,0.013,0.63398



First fixed: q04
Overall R^2: 0.60419


depriv_quantile_min,less65,65-70,70-75,75-80,over80,r2
str,f64,f64,f64,f64,f64,f64
"""q00""",0.0004,0.005,0.005,0.007,0.015,0.50965
"""q02""",0.0004,0.003,0.003,0.007,0.014,0.58367
"""q04""",0.0003,0.002,0.002,0.007,0.014,0.64668
"""q06""",0.0003,0.002,0.002,0.006,0.013,0.62468
"""q08""",0.0002,0.002,0.002,0.005,0.013,0.63577



First fixed: q06
Overall R^2: 0.60419


depriv_quantile_min,less65,65-70,70-75,75-80,over80,r2
str,f64,f64,f64,f64,f64,f64
"""q00""",0.0004,0.005,0.005,0.007,0.015,0.50965
"""q02""",0.0004,0.003,0.003,0.007,0.014,0.58367
"""q04""",0.0003,0.002,0.002,0.007,0.014,0.64668
"""q06""",0.0003,0.002,0.002,0.006,0.013,0.62468
"""q08""",0.0002,0.002,0.002,0.005,0.013,0.63577



First fixed: q08
Overall R^2: 0.60386


depriv_quantile_min,less65,65-70,70-75,75-80,over80,r2
str,f64,f64,f64,f64,f64,f64
"""q00""",0.0004,0.006,0.006,0.006,0.014,0.51042
"""q02""",0.0004,0.003,0.004,0.006,0.014,0.58385
"""q04""",0.0003,0.003,0.003,0.005,0.014,0.64526
"""q06""",0.0002,0.003,0.003,0.005,0.013,0.62285
"""q08""",0.0001,0.003,0.003,0.004,0.013,0.63672


Compare the overall r-squared scores for the five options:

In [140]:
pl.DataFrame(
    np.vstack((
        quantile_str_list,
        [best_picked_combos_r2s[quantile_str][-1] for quantile_str in quantile_str_list]
    )).T,
    schema=['fixed_depriv_quantile_min', 'r2_all']
)

fixed_depriv_quantile_min,r2_all
str,str
"""q00""","""0.60357"""
"""q02""","""0.60357"""
"""q04""","""0.60419"""
"""q06""","""0.60419"""
"""q08""","""0.60386"""


The highest overall R-squared is for the sets of coefficients that first fixed the 0.4-0.6 and 0.6-0.8 deprivation quantiles.

## Compare results with SSNAP-derived coefficients

In [7]:
df_best_coeffs = pl.read_csv(os.path.join('outputs', 'coeffs_prob_stroke_given_age_depriv.csv'))

In [8]:
df_best_coeffs

depriv_quantile_min,less65,65-70,70-75,75-80,over80,r2
str,f64,f64,f64,f64,f64,f64
"""q00""",0.0004,0.005,0.005,0.007,0.015,0.50965
"""q02""",0.0004,0.003,0.003,0.007,0.014,0.58367
"""q04""",0.0003,0.002,0.002,0.007,0.014,0.64668
"""q06""",0.0003,0.002,0.002,0.006,0.013,0.62468
"""q08""",0.0002,0.002,0.002,0.005,0.013,0.63577


In [16]:
coeffs_ssnap

array([0.000408, 0.002644, 0.003735, 0.006005, 0.011558])

In [15]:
df_best_coeffs[['less65', '65-70', '70-75', '75-80', 'over80']].to_numpy() / coeffs_ssnap

array([[0.98039216, 1.89107413, 1.33868809, 1.16569525, 1.29780239],
       [0.98039216, 1.13464448, 0.80321285, 1.16569525, 1.21128223],
       [0.73529412, 0.75642965, 0.53547523, 1.16569525, 1.21128223],
       [0.73529412, 0.75642965, 0.53547523, 0.99916736, 1.12476207],
       [0.49019608, 0.75642965, 0.53547523, 0.83263947, 1.12476207]])

## Calculate r-squared

In [39]:
def calculate_rsquared(y, yhat):
    """This gives the same results as the sklearn built-in."""
    y = np.array(y)
    yhat = np.array(yhat)
    y_mean = np.mean(y)#.mean()
    ss_res = np.sum(((yhat - y)**2.0))#.sum()
    ss_tot = np.sum(((y - y_mean)**2.0))#.sum()
    if ss_tot != 0.0:
        rsq = 1.0 - ss_res / ss_tot
    else:
        rsq = np.NaN
    return rsq

In [40]:
def calculate_many_rsquared(coeffs, x_lists, coeffs_ssnap, admissions_lists):
    list_r2 = []
    predictions_lists = []
    for i in range(5):
        # Population numbers for areas in this quantile:
        x_lists_here = x_lists[i]
        # Separate prediction for each deprivation quantile:
        c = np.array(coeffs[(i*5):(i*5)+5]) #* np.array(coeffs_ssnap)
        yhat = predict_admissions(x_lists_here, c)
        predictions_lists.append(yhat)
        observed_here = admissions_lists[i]
        r2 = calculate_rsquared(observed_here, yhat)
        list_r2.append(r2)

    # Combine lists:
    observed_all = sum(admissions_lists, [])
    predicted_all = sum(predictions_lists, [])
    # Calculate r-squared:
    r2 = calculate_rsquared(observed_all, predicted_all)
    list_r2.append(r2)
    return list_r2

Check results:

In [20]:
df_best_gens.sort('r2_all', descending=True)

dir,# gen,age_less65_q00,age_65_q00,age_70_q00,age_75_q00,age_over80_q00,age_less65_q02,age_65_q02,age_70_q02,age_75_q02,age_over80_q02,age_less65_q04,age_65_q04,age_70_q04,age_75_q04,age_over80_q04,age_less65_q06,age_65_q06,age_70_q06,age_75_q06,age_over80_q06,age_less65_q08,age_65_q08,age_70_q08,age_75_q08,age_over80_q08,fitness,r2_all,r2_q00,r2_q02,r2_q04,r2_q06,r2_q08
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""randomseed24""",23.0,0.0005,0.004,0.005,0.007,0.014,0.0004,0.003,0.004,0.007,0.013,0.0004,0.002,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,238.421,0.593483,0.509119,0.583466,0.634746,0.612783,0.606993
"""randomseed08""",36.0,0.0005,0.003,0.005,0.007,0.015,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.011,0.0004,0.003,0.004,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,238.56,0.591779,0.508469,0.583466,0.631482,0.608507,0.606993
"""randomseed04""",39.0,0.0005,0.003,0.005,0.007,0.015,0.0004,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.011,0.0004,0.002,0.003,0.005,0.011,0.0003,0.002,0.003,0.005,0.01,238.453,0.591769,0.508469,0.579147,0.631482,0.612783,0.606993
"""randomseed17""",20.0,0.0005,0.003,0.005,0.007,0.015,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.011,0.0004,0.003,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,239.924,0.591412,0.508469,0.583466,0.631482,0.606683,0.606993
"""randomseed19""",34.0,0.0005,0.003,0.005,0.007,0.015,0.0004,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.011,0.0004,0.003,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,239.009,0.591412,0.508469,0.583466,0.631482,0.606683,0.606993
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""randomseed65""",19.0,0.0005,0.004,0.005,0.008,0.014,0.0005,0.003,0.004,0.007,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,239.802,0.57744,0.505377,0.556782,0.616005,0.583824,0.606993
"""randomseed05""",19.0,0.0005,0.003,0.004,0.007,0.015,0.0004,0.003,0.004,0.006,0.012,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,238.254,0.577355,0.502059,0.559312,0.616005,0.583824,0.606993
"""randomseed89""",25.0,0.0005,0.003,0.004,0.007,0.014,0.0005,0.003,0.004,0.006,0.013,0.0004,0.003,0.004,0.006,0.012,0.0004,0.002,0.003,0.005,0.01,0.0003,0.002,0.003,0.005,0.01,238.894,0.576409,0.485431,0.569427,0.616005,0.583824,0.606993


## Save results

Keep a copy of the picked coeff combo that had the highest overall r-squared.

In [ ]:
best_picked_combos_dfs['q06']

In [110]:
best_picked_combos_dfs['q06'].write_csv(os.path.join('outputs', 'coeffs_prob_stroke_given_age_depriv.csv'))